In [1]:
# Install PySpark in Google Colab
!pip install pyspark

# Import necessary modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, size, array_contains, explode, sort_array, concat, lit, array, transform
from pyspark.sql.functions import map_keys, map_values, map_entries, map_concat, create_map

# Create Spark Session
spark = SparkSession.builder \
    .appName("Spark Array and Map Operations") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.1


In [2]:
# 1. Create DataFrame with Array Column
# -----------------------------
data_array = [
    (1, ["apple", "banana", "orange"]),
    (2, ["mango", "grape"]),
    (3, ["kiwi", "melon", "apple"])
]
df_array = spark.createDataFrame(data_array, ["id", "fruits"])

print("\nOriginal DataFrame with Array Column:")
df_array.show(truncate=False)

# Array Functions
df_array_ops = df_array.select(
    col("id"),
    col("fruits"),
    size(col("fruits")).alias("array_size"),         # Size of array
    array_contains(col("fruits"), "apple").alias("has_apple"),  # Check if array contains 'apple'
    sort_array(col("fruits"), asc=True).alias("sorted_fruits"), # Sort array
    explode(col("fruits")).alias("exploded_fruit")    # Explode array into rows
)

print("\nArray Operations Result:")
df_array_ops.show(truncate=False)


Original DataFrame with Array Column:
+---+-----------------------+
|id |fruits                 |
+---+-----------------------+
|1  |[apple, banana, orange]|
|2  |[mango, grape]         |
|3  |[kiwi, melon, apple]   |
+---+-----------------------+


Array Operations Result:
+---+-----------------------+----------+---------+-----------------------+--------------+
|id |fruits                 |array_size|has_apple|sorted_fruits          |exploded_fruit|
+---+-----------------------+----------+---------+-----------------------+--------------+
|1  |[apple, banana, orange]|3         |true     |[apple, banana, orange]|apple         |
|1  |[apple, banana, orange]|3         |true     |[apple, banana, orange]|banana        |
|1  |[apple, banana, orange]|3         |true     |[apple, banana, orange]|orange        |
|2  |[mango, grape]         |2         |false    |[grape, mango]         |mango         |
|2  |[mango, grape]         |2         |false    |[grape, mango]         |grape         |
|3  

In [3]:
# 2. Create DataFrame with Map Column
# -----------------------------
data_map = [
    (1, {"name": "Alice", "city": "NY"}),
    (2, {"name": "Bob", "city": "LA"}),
    (3, {"name": "Charlie", "city": "Chicago"})
]
df_map = spark.createDataFrame(data_map, ["id", "person_info"])

print("\nOriginal DataFrame with Map Column:")
df_map.show(truncate=False)

# Map Functions
df_map_ops = df_map.select(
    col("id"),
    col("person_info"),
    map_keys(col("person_info")).alias("keys"),        # Extract keys
    map_values(col("person_info")).alias("values"),    # Extract values
    map_entries(col("person_info")).alias("entries"),  # Convert map to array of structs
    map_concat(col("person_info"), create_map(lit("country"), lit("USA"))).alias("extended_map") # Add new key-value
)

print("\nMap Operations Result:")
df_map_ops.show(truncate=False)


Original DataFrame with Map Column:
+---+----------------------------------+
|id |person_info                       |
+---+----------------------------------+
|1  |{name -> Alice, city -> NY}       |
|2  |{name -> Bob, city -> LA}         |
|3  |{name -> Charlie, city -> Chicago}|
+---+----------------------------------+


Map Operations Result:
+---+----------------------------------+------------+------------------+----------------------------------+--------------------------------------------------+
|id |person_info                       |keys        |values            |entries                           |extended_map                                      |
+---+----------------------------------+------------+------------------+----------------------------------+--------------------------------------------------+
|1  |{name -> Alice, city -> NY}       |[name, city]|[Alice, NY]       |[{name, Alice}, {city, NY}]       |{name -> Alice, city -> NY, country -> USA}       |
|2  |{name -> B

In [4]:
# -----------------------------
# 3. Stop Spark Session
# -----------------------------
spark.stop()